In [1]:
#Bài 1. Cài đặt thuật giải AKT vào bài toán 15 puzzle(n>4).
import copy
from heapq import heappush, heappop

rows = [1, 0, -1, 0]
cols = [0, -1, 0, 1]

class priorityQueue:
    def __init__(self):
        self.heap = []
    def push(self, key):
        heappush(self.heap, key)
    def pop(self):
        return heappop(self.heap)
    def empty(self):
        return len(self.heap) == 0

class nodes:
    def __init__(self, parent, mats, empty_tile_posi, cost, level):
        self.parent = parent
        self.mats = mats
        self.empty_tile_posi = empty_tile_posi
        self.cost = cost
        self.level = level

    def __lt__(self, nxt):
        return self.cost < nxt.cost

def calculatorCosts(mats, final, n):
    cost = 0
    final_pos = {}
    for i in range(n):
        for j in range(n):
            final_pos[final[i][j]] = (i, j)

    for i in range(n):
        for j in range(n):
            val = mats[i][j]
            if val != 0:
                final_x, final_y = final_pos[val]
                cost += abs(i - final_x) + abs(j - final_y)
    return cost

def newNodes(mats, empty_tile_posi, new_empty_tile_posi, level, parent, final, n):
    new_mats = copy.deepcopy(mats)
    x1, y1 = empty_tile_posi
    x2, y2 = new_empty_tile_posi
    new_mats[x1][y1], new_mats[x2][y2] = new_mats[x2][y2], new_mats[x1][y1]
    h = calculatorCosts(new_mats, final, n)
    cost = level + h
    return nodes(parent, new_mats, new_empty_tile_posi, cost, level)

def printMatrix(mats, n):
    for i in range(n):
        for j in range(n):
            print("%2d" % mats[i][j], end=" ")
        print()
    print()

def isSafe(x, y, n):
    return 0 <= x < n and 0 <= y < n

def mat_to_tuple(mat):
    return tuple(tuple(row) for row in mat)

def printPath(root, n):
    if root is None:
        return
    printPath(root.parent, n)
    printMatrix(root.mats, n)

def solve_puzzle(initial, empty_tile_posi, final):
    n = len(initial)
    pq = priorityQueue()

    h = calculatorCosts(initial, final, n)
    root = nodes(None, initial, empty_tile_posi, h, 0)
    pq.push(root)

    visited = set()
    while not pq.empty():
        minimum = pq.pop()
        state = mat_to_tuple(minimum.mats)
        if state in visited:
            continue
        visited.add(state)
        if calculatorCosts(minimum.mats, final, n) == 0:
            print("(Số bước: %d)\n" % minimum.level)
            printPath(minimum, n)
            return

        for i in range(4):
            new_x = minimum.empty_tile_posi[0] + rows[i]
            new_y = minimum.empty_tile_posi[1] + cols[i]

            if isSafe(new_x, new_y, n):
                child = newNodes(minimum.mats, minimum.empty_tile_posi, [new_x, new_y], minimum.level + 1, minimum, final, n)
                pq.push(child)

initial = [
    [1, 2, 3, 4],
    [5, 6, 7, 8],
    [9, 10, 0, 12],
    [13, 14, 11, 15]
]
final = [
    [1, 2, 3, 4],
    [5, 6, 7, 8],
    [9, 10, 11, 12],
    [13, 14, 15, 0]
]
empty_tile_posi = [2, 2]

solve_puzzle(initial, empty_tile_posi, final)

(Số bước: 2)

 1  2  3  4 
 5  6  7  8 
 9 10  0 12 
13 14 11 15 

 1  2  3  4 
 5  6  7  8 
 9 10 11 12 
13 14  0 15 

 1  2  3  4 
 5  6  7  8 
 9 10 11 12 
13 14 15  0 



In [2]:
#Bài 2. Cài đặt bài toán người giao hàng theo thuật giải A*.
from heapq import heappush, heappop

class priorityQueue:
    def __init__(self):
        self.heap = []
    def push(self, key):
        heappush(self.heap, key)
    def pop(self):
        return heappop(self.heap)
    def empty(self):
        return len(self.heap) == 0

class TSPNode:
    def __init__(self, parent, current_city, visited_cities, g, h):
        self.parent = parent
        self.current_city = current_city
        self.visited_cities = visited_cities
        self.g = g
        self.h = h
        self.f = g + h

    def __lt__(self, nxt):
        return self.f < nxt.f
def heuristic_tsp(graph, unvisited_cities):
    if not unvisited_cities:
        return 0

    h = 0
    for city in unvisited_cities:
        min_edge = float('inf')
        for neighbor, weight in graph[city].items():
            if weight < min_edge:
                min_edge = weight
        if min_edge != float('inf'):
            h += min_edge
    return h

def printTSPPath(node):
    path = []
    current = node
    while current is not None:
        path.append(current.current_city)
        current = current.parent
    path.reverse()
    print("Hành trình Người giao hàng:", " -> ".join(path))
    print("Tổng chi phí tối ưu (g):", node.g)

def solve_tsp(graph, start_city):
    pq = priorityQueue()
    all_cities = set(graph.keys())
    unvisited = tuple(all_cities - {start_city})
    h_start = heuristic_tsp(graph, unvisited)
    root = TSPNode(None, start_city, (start_city,), 0, h_start)
    pq.push(root)
    visited_states = set()
    while not pq.empty():
        current = pq.pop()
        state = (current.current_city, tuple(sorted(current.visited_cities)))
        if state in visited_states:
            continue
        visited_states.add(state)
        if len(current.visited_cities) == len(all_cities) + 1 and current.current_city == start_city:
            printTSPPath(current)
            return
        if len(current.visited_cities) == len(all_cities):
            if start_city in graph[current.current_city]:
                weight = graph[current.current_city][start_city]
                new_visited = current.visited_cities + (start_city,)
                child = TSPNode(current, start_city, new_visited, current.g + weight, 0)
                pq.push(child)
            continue
        for neighbor, weight in graph[current.current_city].items():
            if neighbor not in current.visited_cities:
                new_visited = current.visited_cities + (neighbor,)
                unvisited = tuple(all_cities - set(new_visited))

                new_g = current.g + weight
                new_h = heuristic_tsp(graph, unvisited)

                child = TSPNode(current, neighbor, new_visited, new_g, new_h)
                pq.push(child)
tsp_graph = {
    'A': {'B': 10, 'C': 15, 'D': 20},
    'B': {'A': 10, 'C': 35, 'D': 25},
    'C': {'A': 15, 'B': 35, 'D': 30},
    'D': {'A': 20, 'B': 25, 'C': 30}
}
start = 'A'
solve_tsp(tsp_graph, start)

Hành trình Người giao hàng: A -> C -> D -> B -> A
Tổng chi phí tối ưu (g): 80
